Behavioral features are stronger confounders (larger SMDs) so they take priority in W. Demographics go in X where their role is to explain heterogeneity. 

In [0]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import cross_val_score

gold_df = spark.table("causal_project.gold_household").toPandas()
print(f"Shape: {gold_df.shape}")

In [0]:
gold_df['log_outcome'] = np.log1p(gold_df['avg_daily_campaign_spend'])

gold_encoded = gold_df.copy()

gold_encoded['age_group'] = gold_encoded['age_group'].map({
    'Age Group1': 1, 'Age Group2': 2, 'Age Group3': 3,
    'Age Group4': 4, 'Age Group5': 5, 'Age Group6': 6
})
gold_encoded['income_level'] = gold_encoded['income_level'].str.replace(
    'Level', '', regex=False
).astype(int)

gold_encoded['household_size'] = gold_encoded['household_size'].map({
    '1': 1, '2': 2, '3': 3, '4': 4, '5+': 5
})
gold_encoded['kid_count'] = gold_encoded['kid_count'].map({
    'None/Unknown': 0, '1': 1, '2': 2, '3+': 3
})
gold_encoded['home_ownership'] = gold_encoded['home_ownership'].map({
    'Renter': 0, 'Probable Renter': 1, 'Unknown': 2,
    'Probable Owner': 3, 'Homeowner': 4
})
gold_encoded['marital_status'] = gold_encoded['marital_status'].map({
    'X': 0, 'Y': 1, 'Z': 2
})


In [0]:
x_cols = [
    'age_group', 'marital_status', 'income_level',
    'household_size', 'home_ownership', 'kid_count'
]
w_cols = [
    'pre_avg_weekly_spend',
    'pre_avg_weekly_trips',
    'pre_coupon_usage_rate',
    'pre_loyalty_card_rate',
    'pre_department_breadth',
    'pre_active_weeks',
    'clean_window_days',
]

Y = gold_encoded['log_outcome'].values
T = gold_encoded['treatment'].values
X = gold_encoded[x_cols].values.astype(float)
W = gold_encoded[w_cols].values.astype(float)

print(f"Y: {Y.shape}, T: {T.shape}, X: {X.shape}, W: {W.shape}")
print(f"TypeA: {T.sum()}, TypeBC: {(1-T).sum()}")

In [0]:
mlflow.set_experiment("/Users/kayvan2024@gmail.com/causal_project")
with mlflow.start_run(run_name="CausalForestDML_v1"):
    model = CausalForestDML(
        model_y=GradientBoostingRegressor(n_estimators=200,
        max_depth=4,
        random_state=42),
        model_t=GradientBoostingClassifier(n_estimators=200,
        max_depth=4,
        random_state=42),
        discrete_treatment=True,
        cv=3,
        n_estimators=500,
        min_samples_leaf=20,   # conservative given n=760
        max_features="auto",
        random_state=42
        )
    model.fit(Y,T,X=X,W=W)
    ate_inf = model.ate_inference(X=X)
    summary = ate_inf.summary()
    print(summary)
    
    mlflow.log_params({
        "estimator":       "CausalForestDML",
        "n_estimators":    500,
        "min_samples_leaf": 20,
        "outcome":         "log1p(avg_daily_campaign_spend)",
        "treatment":       "TypeA=1 vs TypeBC=0",
        "n_obs":           len(Y),
        "n_treated":       int(T.sum()),
        "n_control":       int((1-T).sum()),
    })

    ci_lower, ci_upper = ate_inf.conf_int_mean()
    mlflow.log_metric("ate", ate_inf.mean_point)
    mlflow.log_metric("ate_ci_lower", ci_lower)
    mlflow.log_metric("ate_ci_upper", ci_upper)
    mlflow.log_metric("ate_pvalue", ate_inf.pvalue())

In [0]:
cate = model.effect(X)
print(f"Households with positive CATE: {(cate > 0).sum()} / {len(cate)}")
print(f"Mean CATE for positive group: {cate[cate > 0].mean():.3f}")
print(f"Mean CATE for negative group: {cate[cate < 0].mean():.3f}")